# VisionVoice - Part 1: EDA and Data Preparation

This notebook audits the VizWiz validation split, removes noisy annotations, creates a leakage-free train/validation split, and studies caption length and vocabulary coverage before model training.

# 1. Introduction to Image Captioning with VizWiz


## 1.1 Motivation and Model Intuition

In the field of **Computer Vision** and **Natural Language Processing (NLP)**, image captioning represents a bridge between visual understanding and linguistic expression. The primary objective of this project is to develop a deep learning model capable of generating descriptive text for images.

This technology has profound real-world applications in **Assistive Technology**. By automating the description of visual environments, we can build accessibility tools that assist individuals who are blind or have low vision. We are utilizing the **VizWiz dataset**, which is specifically designed to reflect the challenges faced by this community.

**The Encoder-Decoder Architecture**

We implement a two-part **Encoder-Decoder** architecture using **PyTorch**:
1.  **The Encoder (The "Eyes")**: A pre-trained **Convolutional Neural Network (CNN)** (e.g., 'ResNet', 'VGG', or 'Inception'). Its job is to compress raw pixel data into a dense, high-dimensional feature vector representing the image's core visual semantic information.
2.  **The Decoder (The "Voice")**: A sequence-based model such as a **Recurrent Neural Network (RNN)**, **Long Short-Term Memory (LSTM)**, or **Gated Recurrent Unit (GRU)**. It takes the visual features and predicts a sequence of tokens, learning the **conditional probability** of a word given the image context and all previously generated words.

## 1.2 Challenge Definition and Evaluation Metrics

The core task is **Image Captioning**: mapping a high-dimensional image input space to a discrete, sequential text output space.

To evaluate the quality of our generated captions, we use the **BLEU (Bilingual Evaluation Understudy)** metric. BLEU measures the **n-gram overlap** between the machine-generated caption and the five human-provided reference captions. We will track four variations:
*   **BLEU-1**: Measures individual word accuracy (unigrams).
*   **BLEU-2, 3, 4**: Measure the fluency and coherence of phrases (bigrams, trigrams, and 4-grams).

While BLEU provides a quantitative baseline, we also conduct rigorous **Qualitative Analysis** through visual inspection of the model's predictions to identify logical vs. hallucinatory errors.

## 1.3 Dataset Overview: VizWiz-Captions

We are utilizing the **validation** split of the **VizWiz-Captions** dataset. Unlike academic datasets such as MS COCO, VizWiz images are captured by people with visual impairments.

### Dataset Characteristics:
*   **Volume**: 7,750 images.
*   **Annotations**: 38,750 total captions (5 human-generated captions per image).
*   **Real-world Complexity**: These images often feature **poor lighting**, **motion blur**, and **unconventional framing**. These "in-the-wild" artifacts make the captioning task significantly more difficult but highly authentic for accessibility applications.

## 1.4 Project Instructions and Workflow

To maintain **MLOps** best practices, our project follows a strictly phased workflow. This promotes collaboration while ensuring individual accountability for model architecture design.

**Phase 1: Collaborative Data Preparation**

The group executes a shared notebook to handle:
*   Downloading the **VizWiz** dataset.
*   Cleaning and **tokenizing** captions.
*   Building a mapping of words to indices (the **Vocabulary**).
*   Constructing PyTorch `DataLoaders`.

**Phase 2: Independent Modeling (Model 1)**

Each member independently designs, trains, and evaluates their first unique architecture. You are encouraged to experiment with different **CNN backbones** and **RNN variants**.

**Phase 3: Review and Refine (Model 2)**

The group analyzes the results of all "Model 1" versions. Each member then builds a **Refined Architecture**, incorporating the collective insights gained from the initial training runs.


# 2. Initialization


## 2.1 Environment Setup and Logging

In [ ]:
import os
import platform
import sys

print(f'Python : {sys.version}')
print(f'OS     : {platform.system()} {platform.release()}')
print(f'CWD    : {os.getcwd()}')

## 2.2 Libraries

Modern Deep Learning relies on a robust ecosystem of libraries. We utilize **PyTorch** for computational graphs and tensor operations, **NumPy** for numerical processing, and **Pandas** for structured data manipulation. We also implement a **Warning Filter** to suppress non-critical engine warnings, ensuring that our final notebook remains professional and readable for peer review or production handover.

In [ ]:
# Standard Python Utilities
import os, sys, gc, time, json, random, zipfile, warnings, shutil
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Iterable, Any
from collections import Counter

# Networking
import requests

# Data Processing
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Image Processing
from PIL import Image, UnidentifiedImageError

# Visualization
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.models as models
import torchvision.transforms as transforms

# NLP & Progress
import nltk
from nltk.translate.bleu_score import corpus_bleu
from tqdm.notebook import tqdm

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# Centralized Device Configuration
# This ensures we have a single source of truth for hardware acceleration
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Target Device: {DEVICE}")

## 2.3 Centralized Configuration (Settings)

A best practice in high-level Data Science is the **Singleton Configuration Pattern**. Instead of hardcoding hyperparameters like 'LEARNING_RATE' or 'BATCH_SIZE' inside functions, we centralize them in a `Settings` class. This facilitates **Hyperparameter Optimization (HPO)** and makes switching between 'train' and 'inference' modes seamless. We use `pathlib` for **Object-Oriented Filesystem Paths**, which is more robust than standard string manipulation.

In [ ]:
class Settings:
    """Centralized configuration class for the Image Captioning project.

    Contains all hyperparameters, environment settings, and dynamic path
    routing for both Kaggle and local VS Code environments.
    """

    # 1. Operational Toggles
    MODE: str = "train"   # "train" or "inference"
    DEBUG: bool = False
    DEBUG_SAMPLE_SIZE: int = 50 # Number of files to audit when DEBUG is True

    # 2. Environment Detection
    IS_KAGGLE: bool = Path("/kaggle/input").exists()

    # 3. Dynamic Path Management
    # Kaggle mounts datasets by slug, for example /kaggle/input/vizwiz.
    # Override these if needed: VIZWIZ_DATA_DIR, VIZWIZ_ANNOTATIONS_DIR, VIZWIZ_IMAGE_DIR.
    if IS_KAGGLE:
        input_root = Path("/kaggle/input")
        discovered_jsons = sorted(input_root.rglob("annotations/annotations/val.json"))
        discovered_jsons += sorted(input_root.rglob("annotations/val.json"))

        if "VIZWIZ_DATA_DIR" in os.environ:
            DATA_DIR = Path(os.environ["VIZWIZ_DATA_DIR"])
        elif discovered_jsons:
            DATA_DIR = discovered_jsons[0].parents[2]
        else:
            candidates = sorted(str(path) for path in input_root.rglob("val.json"))
            raise FileNotFoundError(
                "Could not locate VizWiz val.json. Found candidates: "
                + (", ".join(candidates[:10]) if candidates else "none")
            )

        ANNOTATIONS_DIR = Path(os.getenv("VIZWIZ_ANNOTATIONS_DIR", str(DATA_DIR / "annotations" / "annotations")))

        default_image_dir = DATA_DIR / "val" / "val"
        if not default_image_dir.exists() and (DATA_DIR / "val").exists():
            default_image_dir = DATA_DIR / "val"

        TRAIN_IMG_DIR = Path(os.getenv("VIZWIZ_IMAGE_DIR", str(default_image_dir)))
        VAL_IMG_DIR = TRAIN_IMG_DIR
        WORK_DIR = Path("/kaggle/working")
    else:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            DATA_DIR = Path(os.getenv("VIZWIZ_LOCAL_DATA_DIR", "/content/drive/MyDrive/Sem3_2026 Autumn/94691 Deep Learning/dl_assignments/dl_at3"))
        except ImportError:
            DATA_DIR = Path(os.getenv("VIZWIZ_LOCAL_DATA_DIR", "./data"))

        ANNOTATIONS_DIR = DATA_DIR / "annotations"
        TRAIN_IMG_DIR = DATA_DIR / "images"
        VAL_IMG_DIR = DATA_DIR / "images"
        WORK_DIR = DATA_DIR / "working"

    BASE: Path = DATA_DIR
    CACHE_DIR: Path = WORK_DIR / "cache"
    WORK_CACHE_DIR: Path = CACHE_DIR / "work"

    # /kaggle/input/models/tuannm3823/visionvoice-image-captioning/pytorch/attention-resnet-lstm/1/vision_voice_attention_best.pth
    # /kaggle/input/models/tuannm3823/visionvoice-image-captioning/pytorch/baseline-resnet-lstm/1/vision_voice_baseline_best.pth

    # 4. Remote Data URLs (For Section 3.1)
    URL_ANNOTATIONS: str = "https://vizwiz.cs.colorado.edu/VizWiz_final/annotations/val.json"
    URL_IMAGES: str = "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip"

    # 5. Data Processing & Pipeline Parameters
    SEED: int = 42
    VAL_SPLIT_SIZE: float = 0.2    # 20% of data used for validation
    VOCAB_MIN_FREQ: int = 2        # Minimum word occurrences to be indexed
    IMAGE_SIZE: int = 224          # ResNet standard input size
    NUM_WORKERS: int = 2           # DataLoader subprocesses

    # 6. Model Hyperparameters
    LEARNING_RATE: float = 1e-4
    BATCH_SIZE: int = 32
    EPOCHS: int = 10
    MAX_SEQ_LEN: int = 25          # Max words generated during inference
    EMBED_SIZE: int = 256          # Dimension of shared visual/textual space
    HIDDEN_SIZE: int = 512         # Number of units in LSTM hidden state
    NUM_LAYERS: int = 1            # Number of LSTM layers stacked
    MAX_GRAD_NORM: float = 5.0     # Clipping threshold for vanishing/exploding gradients

    # 7. Evaluation Parameters
    BLEU_EVAL_SAMPLES: int = 500   # Number of validation images to test for BLEU scoring

# Instantiate configuration object
CFG = Settings()

# Post-instantiation directory setup
CFG.CACHE_EXISTS = CFG.CACHE_DIR.exists()
CFG.WORK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODE         : {CFG.MODE}')
print(f'BASE         : {CFG.BASE}')
print(f'CACHE_EXISTS : {CFG.CACHE_EXISTS}')
print(f'CACHE_DIR    : {CFG.CACHE_DIR}')

## 2.4 Memory Optimization and Management

Deep Learning models are notorious for **RAM/VRAM** exhaustion. To mitigate **Out-of-Memory (OOM)** errors and ensure smooth execution, we implement two critical strategies:
1.  **Garbage Collection**: Manually triggering `gc.collect()` and `torch.cuda.empty_cache()` to proactively free up unused memory pointers.
2.  **Data Type Downcasting**: The `reduce_mem_usage` function iterates through a **Pandas DataFrame** and intelligently converts data types to their smallest possible representation (e.g., `float64` to `float32` or `int64` to `int8`). This significantly reduces the memory footprint of large datasets without losing critical precision.

In [ ]:
def clear_memory() -> None:
    """Forces garbage collection and clears VRAM cache to prevent OOM errors."""
    gc.collect()

    # Clear CUDA cache if using NVIDIA
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Clear MPS cache if using Apple Silicon
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

    if CFG.DEBUG:
        print("[Memory] Garbage collection forced and accelerator cache cleared.")


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """
    Iterates through all columns of a dataframe and downcasts data types
    to their smallest possible representation to reduce memory footprint.
    """
    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        # Exclude object (string) and categorical types from numeric downcasting
        if col_type != object and not pd.api.types.is_categorical_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2

    if CFG.DEBUG:
        reduction = 100 * (start_mem - end_mem) / start_mem
        print(f"[Memory] DataFrame compressed from {start_mem:.2f}MB to {end_mem:.2f}MB ({reduction:.1f}% reduction).")

    return df

## 2.5 Seeding

Deep learning is inherently **stochastic**. Weights are initialized randomly, and data is shuffled. To ensure that our performance metrics are comparable across different runs and by different team members, we must enforce **Global Determinism**. We do this by locking the seeds for the Python `random` module, `numpy`, and `torch`. Crucially, we also disable the **cuDNN benchmark** and set it to deterministic mode to ensure identical results on GPU backends.

In [ ]:
def seed_everything(seed: int = CFG.SEED) -> None:
    """
    Locks all random number generators to a specific seed to ensure
    Global Determinism across independent experimental runs.
    """
    # 1. Python and Data structures
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 2. NumPy
    np.random.seed(seed)

    # 3. PyTorch Core
    torch.manual_seed(seed)

    # 4. PyTorch CUDA (Hardware Acceleration)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # Ensures determinism for multi-GPU setups

        # Enforce deterministic convolutions.
        # Note: This may slightly reduce training speed, but guarantees 100% reproducible results.
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f"Global seed successfully locked to: {seed}")

# Execute initialization
seed_everything(CFG.SEED)

# 3. Data Preparing


## 3.1 Data Loading and Preparation

In modern **MLOps pipelines**, manual data handling is a significant bottleneck and a primary source of environment drift. To ensure **reproducibility** and scalability, we must automate the acquisition and initial preparation of our raw assets. The **VizWiz dataset** consists of high-resolution images and complex JSON annotations, which are often too large to fit entirely into system memory (RAM) during the download phase.

Our approach employs **chunked streaming** to download large ZIP files efficiently in small, manageable pieces (typically 8KB to 1MB). This prevents **Out-of-Memory (OOM)** errors during download. After downloading, the function automates the **extraction** of image and annotation files into their designated directories. This structured approach ensures that downstream preprocessing steps have a consistent and predictable data source, whether running on Kaggle or a local Colab environment.

In [ ]:
def download_file(url: str, dest_dir: Path, filename: str = None) -> Path:
    """
    Downloads a file from a URL using chunked streaming to prevent OOM errors.

    Args:
        url (str): The source URL.
        dest_dir (Path): The local directory to save the file.
        filename (str, optional): Custom filename. Defaults to URL's trailing name.

    Returns:
        Path: The absolute path to the downloaded file.
    """
    dest_dir.mkdir(parents=True, exist_ok=True)
    filename = filename or url.split("/")[-1]
    file_path = dest_dir / filename

    if file_path.exists():
        print(f"[Network] '{filename}' already exists in cache. Skipping download.")
        return file_path

    print(f"[Network] Initiating stream for '{filename}'...")
    try:
        # Stream=True prevents loading the entire file into RAM at once
        with requests.get(url, stream=True, timeout=15) as response:
            response.raise_for_status()
            with open(file_path, "wb") as file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
        print(f"[Network] Successfully downloaded: {file_path}")
    except requests.exceptions.RequestException as e:
        print(f"[Error] Failed to download {filename}: {e}")
        raise SystemExit("Critical data missing. Pipeline halted.")

    return file_path

def prepare_dataset() -> None:
    """
    Orchestrates the data pipeline: routing Kaggle paths or downloading/extracting
    the VizWiz dataset into the local Colab environment.
    """
    if CFG.IS_KAGGLE:
        print("[Pipeline] Kaggle environment detected. Utilizing read-only mounted datasets.")
        if not (CFG.ANNOTATIONS_DIR / "val.json").exists():
             print(f"[Warning] val.json not found in {CFG.ANNOTATIONS_DIR}.")
        return

    print("--- Initializing Local Data Pipeline ---")

    # 1. Acquire Annotations (Metadata)
    CFG.ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
    download_file(
        url=CFG.URL_ANNOTATIONS,
        dest_dir=CFG.ANNOTATIONS_DIR,
        filename="val.json"
    )

    # 2. Acquire and Extract Images
    # We verify extraction success by checking if the directory exists and is populated
    if CFG.TRAIN_IMG_DIR.exists() and any(CFG.TRAIN_IMG_DIR.iterdir()):
        print(f"[Pipeline] Image payload verified at: {CFG.TRAIN_IMG_DIR}. Skipping extraction.")
    else:
        print(f"[Pipeline] Image directory empty. Beginning extraction protocol...")
        CFG.TRAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)

        # Download the heavy zip file to the temporary cache
        zip_path = download_file(CFG.URL_IMAGES, CFG.WORK_CACHE_DIR)

        print(f"[Storage] Unpacking compressed images to {CFG.TRAIN_IMG_DIR}...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                # Extract to a temp cache folder to handle unexpected internal nested folders
                temp_extract_path = CFG.WORK_CACHE_DIR / "temp_extract"
                temp_extract_path.mkdir(exist_ok=True)
                zip_ref.extractall(temp_extract_path)

                # Flatten the directory structure if the zip contained a root 'val' folder
                extracted_items = list(temp_extract_path.iterdir())
                if len(extracted_items) == 1 and extracted_items[0].is_dir():
                    source_dir = extracted_items[0]
                else:
                    source_dir = temp_extract_path

                # Move files to final persistent directory
                for item in source_dir.iterdir():
                    shutil.move(str(item), str(CFG.TRAIN_IMG_DIR / item.name))

                # Housekeeping: Clean up temporary extraction folders
                shutil.rmtree(temp_extract_path)

            print("[Storage] Extraction and cleanup complete.")

        except zipfile.BadZipFile:
            print(f"[Error] Corrupted archive detected at {zip_path}.")
            zip_path.unlink() # Delete corrupted zip to force fresh download on next run
            raise SystemExit("Please re-run the cell to attempt download again.")

# Execute Pipeline
prepare_dataset()

## 3.2 Parsing and Cleaning Annotations

In Vision-Language (VL) tasks, the bridge between visual features and linguistic tokens is the **Metadata Annotation**. The VizWiz dataset follows a relational JSON structure (similar to COCO) where image metadata and textual captions are stored in separate lists linked by unique identifiers.

In [ ]:
print(f"--- Parsing Raw Annotations ---")
json_path = CFG.ANNOTATIONS_DIR / "val.json"

# 1. Load the Raw Data
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

images_df = pd.DataFrame(data['images'])
annotations_df = pd.DataFrame(data['annotations'])

# 2. Basic Text Cleanup (Fix legacy carriage returns that break tokenizers)
annotations_df['caption'] = annotations_df['caption'].str.replace('\r', '', regex=False)

# 3. Initial Merge (Attach file_names to captions for Visualization & EDA)
raw_corpus_df = pd.merge(
    annotations_df,
    images_df[['id', 'file_name']],
    left_on='image_id',
    right_on='id',
    how='inner'
)
# Rename the annotation's 'id' column to avoid confusion, drop the duplicate image 'id'
raw_corpus_df = raw_corpus_df.drop(columns=['id_y']).rename(columns={'id_x': 'annotation_id'})

print(f"Successfully loaded {len(raw_corpus_df)} raw annotations.\n")

**3.2.1 Loading Raw Data and Initial Visualization**

Before modifying the dataset, we must ingest the **raw JSON annotations** and inspect them. The VizWiz dataset is structured **relationally**, meaning the image metadata and the actual text captions are stored in separate JSON arrays.

In this step, we:
*   **Parse the JSON** and load it into **Pandas DataFrames**.
*   Perform a preliminary merge to link the `file_name` to its respective captions.
*   Utilize **Plotly** to visually inspect a random sample of the raw data, specifically highlighting captions that crowd-workers flagged as `[REJECTED]` (spam) or `[PRECANNED]` (quality issues).

In [ ]:
def show_raw_sample_with_plotly(df: pd.DataFrame, image_dir: Path, num_samples: int = 1) -> None:
    """Displays random images and color-codes noisy/spam captions for raw data inspection."""
    print(f"--- [Visualization] Sampling {num_samples} raw images ---")
    grouped = df.groupby('file_name')
    unique_files = list(grouped.groups.keys())

    safe_num_samples = min(num_samples, len(unique_files))
    samples = random.sample(unique_files, k=safe_num_samples)

    for file_name in samples:
        group = grouped.get_group(file_name)
        img_path = image_dir / file_name

        captions: List[str] = []
        colors: List[str] = []

        # Process Annotations and Apply Color Coding for Bad Data
        for _, row in group.iterrows():
            is_rejected = row.get('is_rejected', 0.0)
            is_precanned = row.get('is_precanned', 0.0)

            if is_rejected:
                captions.append(f"[REJECTED] {row['caption']}")
                colors.append('crimson')
            elif is_precanned:
                captions.append(f"[PRECANNED] {row['caption']}")
                colors.append('royalblue')
            else:
                captions.append(row['caption'])
                colors.append('black')

        # Setup Plotly Subplot
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.4, 0.6],
            specs=[[{"type": "image"}, {"type": "table"}]],
            horizontal_spacing=0.05
        )

        # Inject Image
        if img_path.exists():
            try:
                with Image.open(img_path) as img:
                    img_array = np.array(img.convert("RGB"))
                    fig.add_trace(go.Image(z=img_array), row=1, col=1)
            except Exception as e:
                print(f"[Warning] Could not render image {file_name}: {e}")

        # Inject Caption Table
        fig.add_trace(
            go.Table(
                header=dict(
                    values=[f"<b>Raw Captions for {file_name}</b>"],
                    align="left",
                    font=dict(color="white", size=14),
                    fill_color="#2C3E50"
                ),
                cells=dict(
                    values=[captions],
                    align="left",
                    font=dict(color=[colors], size=13),
                    height=40,
                    fill_color="#F8F9F9"
                )
            ),
            row=1, col=2
        )

        fig.update_layout(height=400, margin=dict(l=10, r=10, t=30, b=10), plot_bgcolor="white")
        fig.update_xaxes(visible=False, row=1, col=1)
        fig.update_yaxes(visible=False, row=1, col=1)
        fig.show()

# Execute Visualization
if CFG.MODE == "train":
    # Using a seed before sampling ensures we get the same random images every time we run the cell
    random.seed(CFG.SEED)
    show_raw_sample_with_plotly(raw_corpus_df, CFG.TRAIN_IMG_DIR, num_samples=3)

**3.2.2 Exploratory Data Analysis (Dataset Quality)**

Before we perform any destructive filtering on our dataset, we must understand the **distribution of noise**. The VizWiz dataset is highly authentic, meaning it contains unreadable images (flagged as `[PRECANNED]`) and conversational spam (flagged as `[REJECTED]`).

By visualizing the **annotation quality** and the **distribution of "clean" captions** per image, we ensure that our downstream filtering logic does not inadvertently create **"orphaned" images** (images with zero valid captions left), which would crash our PyTorch DataLoader.

In [ ]:
print("--- [EDA] Raw Dataset Quality Analysis ---")

# 1. Identify Quality Flags
# Safely handle JSON boolean/float parsing variations
is_p = (raw_corpus_df['is_precanned'] == True) | (raw_corpus_df['is_precanned'] == 1.0)
is_r = (raw_corpus_df['is_rejected'] == True) | (raw_corpus_df['is_rejected'] == 1.0)

# 2. Annotation-Level Intersection Analysis
both_count = (is_p & is_r).sum()
just_precanned = (is_p & ~is_r).sum()
just_rejected = (is_r & ~is_p).sum()
clean_count = (~is_p & ~is_r).sum()

print("\n[Annotation Breakdown]")
print(f"Total Annotations : {len(raw_corpus_df)}")
print(f"Clean/Descriptive : {clean_count}")
print(f"Precanned Only    : {just_precanned}")
print(f"Rejected Only     : {just_rejected}")
print(f"BOTH (P & R)      : {both_count}")

# 3. Image-Level "Good Caption" Distribution
raw_corpus_df['is_good'] = ~is_p & ~is_r
dist_counts = raw_corpus_df.groupby('file_name')['is_good'].sum().value_counts().sort_index()

# 4. Visualizations (Viridis Palette)
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=("Raw Annotation Quality", "Descriptive Captions per Image")
)

viridis = px.colors.sequential.Viridis

# Pie Chart
fig.add_trace(go.Pie(
    labels=['Clean', 'Precanned', 'Rejected', 'Both'],
    values=[clean_count, just_precanned, just_rejected, both_count],
    hole=.4,
    marker_colors=[viridis[8], viridis[5], viridis[2], viridis[0]]
), row=1, col=1)

# Bar Chart
fig.add_trace(go.Bar(
    x=dist_counts.index,
    y=dist_counts.values,
    marker=dict(color=dist_counts.index, colorscale='Viridis'),
    text=dist_counts.values,
    textposition='auto'
), row=1, col=2)

fig.update_layout(height=450, showlegend=True, plot_bgcolor="white", title_text="VizWiz Quality Breakdown")
fig.update_xaxes(title_text="Valid Captions (0 to 5)", row=1, col=2, showgrid=True, gridcolor='lightgrey', dtick=1)
fig.update_yaxes(title_text="Number of Images", row=1, col=2, showgrid=True, gridcolor='lightgrey')
fig.show()

# Cleanup temporary EDA column
raw_corpus_df.drop(columns=['is_good'], inplace=True)

### Key EDA Insights

*   Exploratory Data Analysis revealed that while 85.5% of the VizWiz annotations were descriptive, 12% consisted of precanned 'quality issue' flags indicative of the dataset's in-the-wild nature.
*   Furthermore, filtering out this noise resulted in 208 completely orphaned images (0 valid captions) and a variable number of reference captions (1-4) for thousands of others.
*   This EDA directly informed our pipeline architecture, necessitating the removal of orphaned images to prevent DataLoader instability and requiring dynamic reference grouping for our BLEU evaluation.

**3.2.3 Data Cleaning and Leakage-Free Splitting**

Having identified the noise in our dataset, we now sterilize the corpus to optimize for our evaluation metric (**BLEU**). Because **BLEU** measures **n-gram overlap**, allowing standard conversational phrases or uniform "Quality issue" strings into the training set will artificially depress our model's scores.

In this step, we:

*   Filter: Drop all `[REJECTED]` and `[PRECANNED]` annotations.

*   Prune Orphans: Drop any images that no longer have valid text targets.

*   Split: Perform an **80/20 Train/Validation split** based on unique images, guaranteeing no **data leakage** between the sets.

In [ ]:
print("--- [Data Filtering] ---")
original_anno_count = len(raw_corpus_df)

# 1. Silently drop Spam and Precanned
clean_corpus_df = raw_corpus_df[
    (raw_corpus_df['is_rejected'] != True) &
    (raw_corpus_df['is_rejected'] != 1.0) &
    (raw_corpus_df['is_precanned'] != True) &
    (raw_corpus_df['is_precanned'] != 1.0)
].reset_index(drop=True)

spam_removed = original_anno_count - len(clean_corpus_df)
print(f"Dropped {spam_removed} noisy annotations.")

# 2. Identify and drop orphaned images (images with 0 valid captions)
# By grouping by file_name, we ensure we only keep images that still exist in clean_corpus_df
valid_images = clean_corpus_df['file_name'].unique()
print(f"Final valid corpus size: {len(clean_corpus_df)} captions across {len(valid_images)} images.\n")

print("--- [Splitting] ---")
# 3. Perform a leakage-free split
train_imgs, val_imgs = train_test_split(
    valid_images,
    test_size=CFG.VAL_SPLIT_SIZE,
    random_state=CFG.SEED
)

train_df = clean_corpus_df[clean_corpus_df['file_name'].isin(train_imgs)].reset_index(drop=True)
val_df = clean_corpus_df[clean_corpus_df['file_name'].isin(val_imgs)].reset_index(drop=True)

# Compress datatypes to save memory
train_df = reduce_mem_usage(train_df)
val_df = reduce_mem_usage(val_df)

print(f"Train Split : {len(train_imgs)} images | {len(train_df)} total captions")
print(f"Val Split   : {len(val_imgs)} images | {len(val_df)} total captions\n")

# 4. Handle Debug Truncation
if CFG.DEBUG:
    print("[Data Prep] DEBUG MODE: Truncating DataFrames for rapid iteration.")
    train_df = train_df.head(100)
    val_df = val_df.head(100)

## 3.2.4 Data and Image Integrity Audits

Before modeling, this section verifies that the cleaned splits are structurally sound and that referenced image files are present and readable. These checks catch common dataset issues early, before a long Kaggle training run fails inside the PyTorch DataLoader.


In [ ]:
def audit_dataset(df: pd.DataFrame, split_name: str) -> None:
    """Print structural quality checks for an image-caption split."""
    print(f"--- Data Quality Audit: {split_name} ---")

    missing_values = int(df.isnull().sum().sum())
    duplicate_ids = int(df['annotation_id'].duplicated().sum()) if 'annotation_id' in df.columns else 0
    empty_captions = int((df['caption'].astype(str).str.strip() == '').sum())
    captions_per_image = df.groupby('file_name').size()

    print(f"Missing values       : {missing_values}")
    print(f"Duplicate annotations: {duplicate_ids}")
    print(f"Empty captions       : {empty_captions}")
    print(f"Unique images        : {df['file_name'].nunique()}")
    print(f"Total captions       : {len(df)}")
    print("\nCaptions per image:")
    print(captions_per_image.value_counts().sort_index().to_string())

    if missing_values == 0 and duplicate_ids == 0 and empty_captions == 0:
        print("\nStatus: passed structural checks.")
    else:
        print("\nStatus: review anomalies before training.")


def audit_image_files(df: pd.DataFrame, image_dir: Path, split_name: str) -> Tuple[List[str], List[str]]:
    """Return missing and corrupted image file names for a split."""
    print(f"--- Image Integrity Audit: {split_name} ---")
    print(f"Image directory: {image_dir}")

    missing_files: List[str] = []
    corrupted_files: List[str] = []

    for file_name in tqdm(df['file_name'].unique(), desc=f"Verifying {split_name} images"):
        img_path = image_dir / file_name
        if not img_path.exists():
            missing_files.append(file_name)
            continue

        try:
            with Image.open(img_path) as img:
                img.verify()
        except (OSError, SyntaxError, UnidentifiedImageError):
            corrupted_files.append(file_name)

    print(f"Checked files   : {df['file_name'].nunique()}")
    print(f"Missing files   : {len(missing_files)}")
    print(f"Corrupted files : {len(corrupted_files)}")

    return missing_files, corrupted_files


def drop_bad_image_rows(df: pd.DataFrame, bad_files: Iterable[str]) -> pd.DataFrame:
    bad_files = set(bad_files)
    if not bad_files:
        return df
    before = len(df)
    cleaned = df[~df['file_name'].isin(bad_files)].reset_index(drop=True)
    print(f"Removed {before - len(cleaned)} annotations linked to missing/corrupted images.")
    return cleaned


audit_dataset(train_df, "Train")
audit_dataset(val_df, "Validation")

missing_train, corrupt_train = audit_image_files(train_df, CFG.TRAIN_IMG_DIR, "Train")
missing_val, corrupt_val = audit_image_files(val_df, CFG.VAL_IMG_DIR, "Validation")

train_df = drop_bad_image_rows(train_df, missing_train + corrupt_train)
val_df = drop_bad_image_rows(val_df, missing_val + corrupt_val)


## 3.3 Building the Vocabulary

In Computer Vision and Natural Language Processing (CV-NLP) tasks like Image Captioning, the model cannot directly interpret raw strings. We must map every unique word to a specific integer representation, a process known as **Numericalization**.

To manage this, we implement a custom `Vocabulary` class. This utility serves several critical MLOps and Data Science purposes:
1.  **Special Token Handling**: We reserve specific IDs for **Special Tokens** such as `<PAD>` (padding sequences to equal length), `<START>`/`<END>` (marking sequence boundaries), and `<UNK>` (Unknown tokens for words not seen during training).
2.  **Frequency Thresholding**: By using a `freq_threshold`, we filter out rare words or typos. This reduces the **Dimensionality** of our embedding layer and prevents the model from overfitting on noise.
3.  **Bidirectional Mapping**: We maintain two dictionaries, 'stoi' (**String-to-Index**) for encoding and 'itos' (**Index-to-String**) for decoding model predictions back into human-readable text.

In [ ]:
class Vocabulary:
    """
    Handles the bidirectional mapping between string tokens and integer IDs.
    """
    def __init__(self, freq_threshold: int = CFG.VOCAB_MIN_FREQ) -> None:
        self.freq_threshold = freq_threshold

        # Initialize dictionaries with reserved MLOps special tokens
        self.itos: Dict[int, str] = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.stoi: Dict[str, int] = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.idx: int = 4

    def __len__(self) -> int:
        """Returns the total number of unique words in the vocabulary."""
        return len(self.itos)

    def tokenize(self, text: str) -> List[str]:
        """Converts raw text to lowercase and isolates punctuation."""
        text = str(text).lower()
        # Ensure punctuation is treated as separate tokens to improve context learning
        for punc in [".", ",", "!", "?", "\"", "'", "(", ")"]:
            text = text.replace(punc, f" {punc} ")
        return text.split()

    def build_vocabulary(self, sentence_list: List[str]) -> None:
        """Populates the stoi and itos dictionaries based on token frequencies."""
        print(f"--- [Vocabulary] Building index with threshold >= {self.freq_threshold} ---")
        frequencies = Counter()

        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1

        # Only add words that meet the frequency requirement to prevent overfitting on typos
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = self.idx
                self.itos[self.idx] = word
                self.idx += 1

        print(f"[Vocabulary] Processed {len(sentence_list)} sentences. Indexed {len(self.itos)} unique tokens.")

    def numericalize(self, text: str) -> List[int]:
        """Converts a raw string into a list of integer tensor indices."""
        tokenized_text = self.tokenize(text)
        return [
            self.stoi.get(token, self.stoi["<UNK>"])
            for token in tokenized_text
        ]

In [ ]:
# 1. Extract all clean training captions
# We ONLY build the vocabulary on the training set to prevent data leakage from the val set
train_captions = train_df['caption'].tolist()

# 2. Instantiate and build the Vocabulary object
vocab = Vocabulary(freq_threshold=CFG.VOCAB_MIN_FREQ)
vocab.build_vocabulary(train_captions)

# 3. Validation Check
if CFG.DEBUG or CFG.MODE == "train":
    # Test the numericalization pipeline on the first available caption
    sample_text = train_captions[0]
    print(f"\n[Validation] Source Text : {sample_text}")
    print(f"[Validation] Tokenized   : {vocab.tokenize(sample_text)}")
    print(f"[Validation] Encoded IDs : {vocab.numericalize(sample_text)}")

In [ ]:
def analyze_nlp_distributions(captions: List[str], vocab: Vocabulary) -> None:
    """
    Analyzes the distribution of caption lengths to inform the MAX_SEQ_LEN hyperparameter,
    and visualizes the most frequent words in the dataset.
    """
    print("--- [EDA] NLP Token Distribution Analysis ---")

    # 1. Calculate Sequence Lengths
    # We add +2 to account for the <START> and <END> tokens we will inject later
    seq_lengths = [len(vocab.tokenize(cap)) + 2 for cap in captions]

    # 2. Calculate Word Frequencies
    all_words = []
    for cap in captions:
        all_words.extend(vocab.tokenize(cap))
    word_counts = pd.Series(all_words).value_counts()

    # 3. Compute the 95th Percentile
    percentile_95 = np.percentile(seq_lengths, 95)

    # 4. Visualizations (Plotly)
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "histogram"}, {"type": "bar"}]],
        subplot_titles=("Caption Length Distribution (Includes Start/End)", "Top 20 Most Frequent Words")
    )

    # Histogram of Sequence Lengths
    fig.add_trace(go.Histogram(
        x=seq_lengths,
        nbinsx=40,
        marker_color='#9B59B6',
        name="Length Count"
    ), row=1, col=1)

    # Add a visual line for the 95th percentile
    fig.add_vline(
        x=percentile_95, line_dash="dash", line_color="red", line_width=2,
        annotation_text=f" 95th Percentile: {int(percentile_95)} words ",
        annotation_position="top right",
        row=1, col=1
    )

    # Bar Chart of Top 20 Words
    top_20 = word_counts.head(20)
    fig.add_trace(go.Bar(
        x=top_20.index,
        y=top_20.values,
        marker=dict(color=top_20.values, colorscale='Viridis'),
        text=top_20.values,
        textposition='outside',
        name="Frequency"
    ), row=1, col=2)

    fig.update_layout(height=450, showlegend=False, plot_bgcolor="white")
    fig.update_xaxes(title_text="Number of Tokens", row=1, col=1, showgrid=True, gridcolor='lightgrey')
    fig.update_yaxes(title_text="Count", row=1, col=1, showgrid=True, gridcolor='lightgrey')
    fig.update_xaxes(tickangle=-45, row=1, col=2)
    fig.show()

    # Console Summary
    print(f"\n[NLP Summary]")
    print(f"Shortest Caption : {min(seq_lengths)} tokens")
    print(f"Longest Caption  : {max(seq_lengths)} tokens")
    print(f"Average Caption  : {int(np.mean(seq_lengths))} tokens")
    print(f"95% Threshold    : {int(percentile_95)} tokens")
    print(f"Actionable Insight: Set CFG.MAX_SEQ_LEN to ~{int(percentile_95)} to capture the vast majority of data without wasting VRAM.")

# Execute the EDA on the clean training captions
if CFG.MODE == "train":
    analyze_nlp_distributions(train_captions, vocab)

### Key NLP Insights:

*   **Insight 1: Optimizing the Sequence Length (`MAX_SEQ_LEN`)**

    Analysis of the tokenized sequence lengths revealed a heavy right-skewed distribution. While the average caption length was only 15 tokens, extreme outliers extended up to 175 tokens. To optimize **VRAM** utilization and training efficiency, we set the model's `MAX_SEQ_LEN` hyperparameter to 25. This covers the 95th percentile of all data (24 tokens), ensuring the model learns complete sentences while avoiding the immense computational waste of padding 95% of batches to accommodate single 175-word outliers.

*   **Insight 2: Vocabulary Composition & Domain Bias**

    Visualizing the most frequent tokens confirmed the expected dominance of English stop words ('a', 'on', 'the'). However, the emergence of specific descriptive nouns and adjectives in the top 20—such as 'white', 'black', 'computer', 'screen', 'table', and 'bottle'—highlights a distinct **domain bias** in the VizWiz dataset toward indoor, desktop, and everyday household environments. This informs our qualitative evaluation expectations, as the model will likely perform best on these heavily represented indoor object classes.